# Caso C — AOV - EDA sobre la tabla maestra

> **Objetivo.** Drivers y predicción del ticket promedio. Cruzo transacciones con loyalty, variables exógenas e intensidad de promociones.

> **Entradas.** Fuentes crudas del caso cargadas por el catálogo de Kedro (`data/01_raw`), sin `pd.read_csv` sueltos.

> **Salidas.** La tabla maestra cruzada, su diagnóstico de cobertura y las conclusiones del EDA (tipado, VIF, correlaciones, información mutua, tests).

> **Cómo ejecutar.** Reinicia el kernel y ejecuta todo de arriba abajo (`Restart & Run All`); es determinista. Reutiliza `tostao_ml` (no reimplementa lógica): el notebook orquesta y narra.

## 1. Construcción de la tabla maestra (cruce de fuentes)

`transacciones ⨝ loyalty(cliente) ⨝ exógenas(fecha+tienda) ⨝ intensidad_promos(fecha+tienda)` — una fila por ticket enriquecida.

In [ ]:
from pathlib import Path
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project
from tostao_ml.cases import masters
from tostao_ml.framework.profiling import profile_dataset, build_eda_figures

PROJECT = Path.cwd().parents[1] if Path.cwd().name.startswith('caso') else Path.cwd()
bootstrap_project(PROJECT)
with KedroSession.create(project_path=PROJECT) as session:
    catalog = session.load_context().catalog
    trans = catalog.load('c_transacciones_resumen'); loy = catalog.load('c_clientes_loyalty')
    exo = catalog.load('c_variables_exogenas'); promo = catalog.load('c_promociones_activas')
master, join_report = masters.build_master_c(trans, loy, exo, promo)


In [ ]:
print('Master:', master.shape)
print('Cobertura de cruces:', join_report)
master.head()

La **cobertura de cruces** confirma la integridad referencial: la proporción de filas del hecho que encontró match en cada fuente unida.

## 2. Perfilado estadístico (motor de EDA reutilizable)

In [ ]:
profile = profile_dataset(master, name='Caso C — AOV', target='total_venta')
print('Tipos:')
for c, k in profile.types.items():
    print(f'  {c:28s} {k.value}')
profile.univariate.round(3)

### Mini-conclusiones autogeneradas (cifras reales del run)

In [ ]:
from IPython.display import Markdown
Markdown(profile.narrative.to_markdown())

## 3. Multicolinealidad y correlaciones

In [ ]:
figs = build_eda_figures(master, profile)
display(profile.vif.round(3).to_frame('VIF'))
figs.get('correlation_heatmap')

## 4. Relación con el target e información mutua

In [ ]:
display(profile.mutual_information.round(4).to_frame('MI'))
profile.bivariate.round(4)

In [ ]:
for name, fig in figs.items():
    if name.startswith(('dist__', 'target__')):
        fig.show()

## 5. Conclusión

El EDA sobre la **tabla maestra cruzada** (no fuente por fuente) revela el tipado de cada variable, la multicolinealidad (VIF), las correlaciones y qué features discriminan el target (tests + información mutua). Estas conclusiones guían el feature engineering y la elección de modelo del caso.